# Concurrency Stress Tests

**Release:** v1.0.0 | **Date:** 2026-03-17

## Test Case
- **Users**: 50 concurrent users  
- **Duration**: 30-min mixed R/W workload  
- **Target**: P99 latency stability under heavy load

## Storage Types Under Test
| Schema | Storage Type | Table |
|--------|--------------|-------|
| MANAGED_ICEBERG | Snowflake Managed | CLAIMS_AUDIT |
| TESTS | Customer Storage | CLAIMS_AUDIT |

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICEBERG_POC;
USE SCHEMA TESTS;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
-- Create CLAIMS_AUDIT table for concurrency testing (Customer Storage)
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.TESTS.CLAIMS_AUDIT (
    claim_id        INT,
    patient_id      INT,
    action          VARCHAR,
    audit_payload   VARIANT,
    created_at      TIMESTAMP_NTZ(9)
)
CATALOG = SNOWFLAKE
EXTERNAL_VOLUME = EXVOL
ICEBERG_VERSION = 3;

-- Create CLAIMS_AUDIT on Managed Storage
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.MANAGED_ICEBERG.CLAIMS_AUDIT (
    claim_id        INT,
    patient_id      INT,
    action          VARCHAR,
    audit_payload   VARIANT,
    created_at      TIMESTAMP_NTZ(9)
)
CATALOG = 'SNOWFLAKE'
ICEBERG_VERSION = 3;

-- Pre-populate Customer Storage with 100K baseline audit records
INSERT INTO ICEBERG_POC.TESTS.CLAIMS_AUDIT
SELECT
    SEQ4() + 4001   AS claim_id,
    SEQ4() % 100000 + 1001 AS patient_id,
    'Submit'         AS action,
    OBJECT_CONSTRUCT(
        'payer',       ARRAY_CONSTRUCT('Blue Cross','Aetna','United Healthcare','Cigna','Humana')[SEQ4() % 5]::VARCHAR,
        'reviewer',    'system',
        'reason_code', ARRAY_CONSTRUCT('INITIAL_SUBMIT','AUTO_ELIGIBLE','PRIOR_AUTH_OK')[SEQ4() % 3]::VARCHAR
    )                AS audit_payload,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ(9) AS created_at
FROM TABLE(GENERATOR(ROWCOUNT => 100000));

-- Pre-populate Managed Storage with same data
INSERT INTO ICEBERG_POC.MANAGED_ICEBERG.CLAIMS_AUDIT
SELECT * FROM ICEBERG_POC.TESTS.CLAIMS_AUDIT;

## Concurrent Read Queries
Run these queries simultaneously from multiple sessions to test read concurrency.

In [ ]:
-- Read Query 1: Full scan aggregation — claim actions by payer (compare both storage types)
SELECT 'Customer' AS storage_type, action, audit_payload:payer::VARCHAR AS payer, COUNT(*) AS cnt
FROM ICEBERG_POC.TESTS.CLAIMS_AUDIT GROUP BY 1, 2, 3 ORDER BY 4 DESC LIMIT 5;

SELECT 'Managed' AS storage_type, action, audit_payload:payer::VARCHAR AS payer, COUNT(*) AS cnt
FROM ICEBERG_POC.MANAGED_ICEBERG.CLAIMS_AUDIT GROUP BY 1, 2, 3 ORDER BY 4 DESC LIMIT 5;

-- Read Query 2: Point lookup by claim_id
SELECT 'Customer' AS storage_type, * FROM ICEBERG_POC.TESTS.CLAIMS_AUDIT WHERE claim_id = 50000
UNION ALL
SELECT 'Managed', * FROM ICEBERG_POC.MANAGED_ICEBERG.CLAIMS_AUDIT WHERE claim_id = 50000;

-- Read Query 3: Range scan of recent audit events
SELECT 'Customer' AS storage_type, COUNT(*) AS cnt FROM ICEBERG_POC.TESTS.CLAIMS_AUDIT WHERE claim_id BETWEEN 10000 AND 20000
UNION ALL
SELECT 'Managed', COUNT(*) FROM ICEBERG_POC.MANAGED_ICEBERG.CLAIMS_AUDIT WHERE claim_id BETWEEN 10000 AND 20000;

-- Read Query 4: VARIANT query — denied claims
SELECT 'Customer' AS storage_type, claim_id, audit_payload:reason_code::VARCHAR AS reason
FROM ICEBERG_POC.TESTS.CLAIMS_AUDIT WHERE claim_id < 100 ORDER BY claim_id LIMIT 5;

## Concurrent Write Operations
Test mixed read/write workloads to measure P99 latency stability.

In [ ]:
-- Write Operation 1: Insert batch of claim audit events (Customer Storage)
INSERT INTO ICEBERG_POC.TESTS.CLAIMS_AUDIT
SELECT
    (SELECT MAX(claim_id) FROM ICEBERG_POC.TESTS.CLAIMS_AUDIT) + SEQ4() AS claim_id,
    SEQ4() % 100000 + 1001  AS patient_id,
    ARRAY_CONSTRUCT('Approve','Deny','Review','Resubmit')[SEQ4() % 4]::VARCHAR AS action,
    OBJECT_CONSTRUCT(
        'payer',       ARRAY_CONSTRUCT('Blue Cross','Aetna','United Healthcare','Cigna','Humana')[SEQ4() % 5]::VARCHAR,
        'reviewer',    ARRAY_CONSTRUCT('system','clinical_staff','payer_portal')[SEQ4() % 3]::VARCHAR,
        'reason_code', ARRAY_CONSTRUCT('AUTH_OK','DENIAL_MEDICAL_NECESSITY','AUTH_EXPIRED','RESUBMIT_CORRECTED')[SEQ4() % 4]::VARCHAR
    ) AS audit_payload,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ(9) AS created_at
FROM TABLE(GENERATOR(ROWCOUNT => 1000));

-- Write Operation 1b: Insert batch of claim audit events (Managed Storage)
INSERT INTO ICEBERG_POC.MANAGED_ICEBERG.CLAIMS_AUDIT
SELECT
    (SELECT MAX(claim_id) FROM ICEBERG_POC.MANAGED_ICEBERG.CLAIMS_AUDIT) + SEQ4() AS claim_id,
    SEQ4() % 100000 + 1001  AS patient_id,
    ARRAY_CONSTRUCT('Approve','Deny','Review','Resubmit')[SEQ4() % 4]::VARCHAR AS action,
    OBJECT_CONSTRUCT(
        'payer',       ARRAY_CONSTRUCT('Blue Cross','Aetna','United Healthcare','Cigna','Humana')[SEQ4() % 5]::VARCHAR,
        'reviewer',    ARRAY_CONSTRUCT('system','clinical_staff','payer_portal')[SEQ4() % 3]::VARCHAR,
        'reason_code', ARRAY_CONSTRUCT('AUTH_OK','DENIAL_MEDICAL_NECESSITY','AUTH_EXPIRED','RESUBMIT_CORRECTED')[SEQ4() % 4]::VARCHAR
    ) AS audit_payload,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ(9) AS created_at
FROM TABLE(GENERATOR(ROWCOUNT => 1000));

-- Write Operation 2: Update audit payload (Customer Storage)
UPDATE ICEBERG_POC.TESTS.CLAIMS_AUDIT
SET audit_payload = OBJECT_INSERT(audit_payload, 'reviewed_at', CURRENT_TIMESTAMP()::VARCHAR),
    action = 'Reviewed'
WHERE claim_id BETWEEN 5000 AND 6000;

-- Write Operation 2b: Update audit payload (Managed Storage)
UPDATE ICEBERG_POC.MANAGED_ICEBERG.CLAIMS_AUDIT
SET audit_payload = OBJECT_INSERT(audit_payload, 'reviewed_at', CURRENT_TIMESTAMP()::VARCHAR),
    action = 'Reviewed'
WHERE claim_id BETWEEN 5000 AND 6000;

## Query History Analysis
Analyze query latencies from concurrent workload.

In [ ]:
-- Analyze query performance for CLAIMS_AUDIT Iceberg table operations
SELECT
    QUERY_TYPE,
    COUNT(*)                                                                 AS query_count,
    AVG(TOTAL_ELAPSED_TIME) / 1000                                           AS avg_latency_sec,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY TOTAL_ELAPSED_TIME) / 1000 AS p95_latency_sec,
    PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY TOTAL_ELAPSED_TIME) / 1000 AS p99_latency_sec,
    MAX(TOTAL_ELAPSED_TIME) / 1000                                           AS max_latency_sec
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE QUERY_TEXT ILIKE '%CLAIMS_AUDIT%'
    AND START_TIME > DATEADD('hour', -1, CURRENT_TIMESTAMP())
GROUP BY QUERY_TYPE
ORDER BY query_count DESC;